# Stage 0b - speed + SAE-scale benchmark (Kaggle GPU T4 x2, Internet ON). 10-20 min. Paste the full output of the last cell.

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/sarrazer24/reins-mini"
if not os.path.isdir("/kaggle/working/reins-mini"):
    !git clone {REPO} /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!pip install -q -U transformers accelerate huggingface_hub
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS      # authors' data + frozen controllers; deleted at the end
GUISE = "/tmp/REINS/guise"; CTRL = "/tmp/REINS/configs/qwen35_2b_guise.controllers.json"
from huggingface_hub import snapshot_download
SAE_ROOT = snapshot_download("Carlos4869/REINS-SAE", allow_patterns=["qwen3.5-2b-base/*"], local_dir="/tmp/sae") + "/qwen3.5-2b-base"
print(os.listdir(SAE_ROOT)[:5])

In [ ]:
# Speed kernels for Qwen3.5's linear-attention layers. The pilot ran 114 s per prompt WITHOUT them.
!pip install -q flash-linear-attention
!timeout 1500 pip install -q causal-conv1d --no-build-isolation || echo "causal-conv1d failed or timed out: continue without it"
import importlib
for mod in ("fla", "causal_conv1d"):
    try: importlib.import_module(mod); print(mod, "OK")
    except Exception as e: print(mod, "missing:", type(e).__name__)

In [ ]:
!python -u scripts/05_benchmark_speed.py --sae-root {SAE_ROOT} --guise-dir {GUISE} --layers 14,20,23

In [ ]:
!rm -rf /tmp/REINS /tmp/sae